# MAAI 902: Assessment 2, NLP Solution Design
## Intelligent Procurement Document Analysis for Spare Parts (Heavy Equipment)

**Student:** Hala Sarhan  |  **Lecturer:** Dr. Pallavi Ranjan  |  University of Wollongong in Dubai

### Problem
Procurement teams handling heavy-equipment spare parts receive a constant stream of unstructured documents: purchase orders, supplier emails, maintenance reports and general correspondence. Sorting them, pulling out part numbers, quantities, suppliers, dates and sites, judging supplier tone and urgency, and finding earlier related documents is slow and error-prone when done by hand.

### What this notebook builds
An end-to-end NLP pipeline that turns raw documents into structured, searchable, and answerable information:

| Stage | Method | Output |
|---|---|---|
| 1. Pre-processing | Cleaning, abbreviation expansion, stopword removal | Normalised text |
| 2. Classification | TF-IDF + 5 classifiers (LR, SVM, NB, RF, GB) | Document type |
| 3. Entity extraction | Regex + gazetteer NER | Parts, suppliers, quantities, dates, sites, equipment |
| 4. Sentiment & urgency | Keyword rules | Supplier tone, order urgency |
| 5. Retrieval | TF-IDF vs dense (MiniLM + FAISS) vs hybrid (RRF) | Most relevant documents |
| 6. Grounded generation | Qwen2.5-3B-Instruct (GPU) with citations and refusal | Answers that cite their sources |
| 7. Demo | Gradio web app | Interactive prototype |

## 1. Environment Setup and Library Installation

This notebook runs on a **Google Colab T4 GPU** (Runtime → Change runtime type → T4 GPU). The cell below installs the extra packages (sentence-transformers, FAISS, transformers, Gradio), fixes the random seed, and confirms the GPU is active. The next cell imports the core libraries.

In [1]:
!pip -q install sentence-transformers faiss-cpu transformers accelerate gradio
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("GPU available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
!nvidia-smi

GPU available: True Tesla T4
Mon Oct 12 07:34:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------

In [2]:
# Install additional libraries (uncomment if running in Colab)

import random
import pandas as pd
import numpy as np
import re
import json
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All libraries imported successfully")

All libraries imported successfully


## 2. Synthetic Procurement Dataset

Real procurement documents are confidential, so the dataset is **synthetic**, generated from templates with controlled vocabulary.

- **550 documents:** 150 purchase orders, 150 supplier emails, 150 maintenance reports, 100 general correspondence.
- **Gold labels** are stored with every document: document type, entities, urgency (purchase orders), and true sentiment (supplier emails). These let us measure real precision, recall and accuracy.
- **9 templates per document type**, each with a `template_id`. Later we train on some templates and test on unseen ones.
- **Realism:** about 25% of documents (137) have OCR-style character noise; about 15% of supplier emails are deliberately "hard" (mixed positive and negative signals); vocabulary overlaps between document types.

**Limitation:** because the text is template-based, results are optimistic compared with real documents. This is why the unseen-template and noisy-text tests matter.


In [3]:
import numpy as np, pandas as pd
SEED = 42
np.random.seed(SEED)

parts = [
    "bearing assembly 6205 2RS", "hydraulic pump HP 200", "conveyor belt CB 500",
    "servo motor SM 400", "gearbox GB 300", "control panel CP 100",
    "air filter AF 250", "power supply unit PSU 750", "sensor module SNS 120",
    "drive shaft DS 600", "cooling fan CF 180", "pressure valve PV 350",
    "transformer TR 900", "brake pad BP 080", "compressor unit CU 450"
]
suppliers = [
    "PartsPro International", "TechSupply Global", "MechaComponents Ltd",
    "PrecisionParts Corp", "IndustrialDirect GmbH", "SpareMax Solutions",
    "GlobalMech Industries", "QualityParts Asia", "EuroParts Distribution"
]
equipment_list = [
    "Conveyor System A1", "Robotic Arm Unit R7", "Sorting Machine SM 3",
    "Packaging Line PL 5", "CNC Machine Tool M2", "HVAC System H4"
]
sites = [
    "Dubai Logistics Hub", "Abu Dhabi Warehouse", "Jebel Ali Free Zone",
    "KIZAD Industrial", "Sharjah Distribution Center"
]
failures = ["excessive vibration", "thermal overload", "mechanical wear",
            "electrical short circuit", "bearing seizure", "corrosion damage"]
categories = ["Mechanical", "Electrical", "Hydraulic", "Pneumatic", "Electronic"]

statuses_positive = ["confirmed and on schedule", "shipped ahead of schedule", "ready for dispatch"]
statuses_negative = ["delayed by 2 weeks", "delayed due to supply shortage",
                     "on hold pending quality check", "backordered until next month"]
statuses_neutral = ["being processed", "under quality inspection", "awaiting final confirmation"]
details_positive = ["All items passed quality inspection", "Shipment includes bonus spare components",
                    "Express shipping arranged at no extra cost"]
details_negative = ["Raw material shortage has affected production",
                    "Customs clearance issues are causing additional delays",
                    "Quality defects found in initial batch requiring rework",
                    "Price increase of 15% effective next month"]
details_neutral = ["Expected completion within standard lead time", "No issues reported at this stage",
                   "Will provide tracking details once dispatched"]

# (tag text, gold urgency level)
urgency_tags = [
    ("Standard delivery timeline acceptable.", "ROUTINE"),
    ("URGENT: Equipment downtime impacting operations. Expedite delivery.", "HIGH"),
    ("CRITICAL: Production line halted. Immediate shipment required.", "CRITICAL"),
    ("Routine replenishment order. No rush.", "ROUTINE"),
    ("High priority: Safety related component. Fast track processing.", "HIGH"),
]
priority_tags = ["Priority: Routine", "Priority: High", "Priority: Critical",
                 "Priority: Emergency", "Priority: Standard"]

# Templates deliberately OVERLAP across classes (shared words, part names,
# suppliers, PO references) so document type cannot be guessed from one keyword.
purchase_orders = [
    "PO {id}: Request for {qty} units of {part}. Supplier: {supplier}. Delivery required by {date}. {urgency}",
    "Purchase Order {id}: Procurement of {qty}x {part} from {supplier}. Expected delivery: {date}. {urgency}",
    "Order Reference PO{id}: Please supply {qty} pieces of {part}. Vendor: {supplier}. Need by {date}. {urgency}",
    "Requisition {id}: {qty} units of {part} needed from {supplier}. Target date: {date}. {urgency}",
    "PO{id}: Ordering {qty} {part} components. Approved vendor: {supplier}. Ship by {date}. {urgency}",
    "Please raise an order for {qty} units of {part} with {supplier} for {site}. Required by {date}. {urgency}",
    "Replenishment request {id} for {site}: {part}, qty: {qty}. Preferred vendor {supplier}. Due {date}. {urgency}",
    "Following the failure on {equipment}, we need {qty} units of {part} from {supplier} by {date}. {urgency}",
    "Buyer note {id}: Reorder {qty} pcs {part}. Source: {supplier}. Delivery date {date}. {urgency}",
]
supplier_emails = [
    "Dear Procurement Team, We wish to inform you that the shipment of {part} has been {status}. {detail}. Regards, {supplier}.",
    "Notification from {supplier}: Your order for {part} is {status}. {detail}. Please contact us for any queries.",
    "Subject: Order Update. The delivery of {part} from {supplier} is {status}. {detail}. We apologize for any inconvenience.",
    "{supplier} update: We regret to inform you that {part} delivery is {status}. {detail}. Alternative arrangements are being explored.",
    "Hello, This is to confirm that your {part} order with {supplier} is {status}. {detail}. Best regards, Sales Team.",
    "Re: PO {id}. Your order for {qty} units of {part} is {status}. {detail}. Kind regards, {supplier}.",
    "Hi team, regarding PO{id} ({qty} pieces of {part}) delivering to {site}: status is {status}. {detail}. {supplier}.",
    "Delivery advice from {supplier}: {qty}x {part} for {site} is {status}. {detail}. Please confirm receipt.",
    "FYI from {supplier}. The {part} consignment ({qty} units) is {status}. {detail}.",
]
maintenance_reports = [
    "Work Order WO{id}: {equipment} at Site {site} requires replacement of {part}. Failure mode: {failure}. {priority}.",
    "Maintenance Report MR{id}: Scheduled maintenance on {equipment} identified worn {part}. Recommended replacement. Site: {site}. {priority}.",
    "Service Report SR{id}: Emergency repair needed for {equipment} at {site}. Component {part} has failed due to {failure}. {priority}.",
    "Inspection Report IR{id}: Routine check on {equipment} revealed degradation in {part}. Site {site}. Action: Replace within 30 days. {priority}.",
    "Technician Note TN{id}: {equipment} downtime at {site} caused by {part} malfunction. Root cause: {failure}. {priority}.",
    "{equipment} stopped at {site}. Cause: {failure} on the {part}. Spare was last supplied by {supplier}. {priority}.",
    "Shift handover: {part} on {equipment} ({site}) showing {failure}. Order of {qty} units to be requested. {priority}.",
    "Breakdown log {id}: {site}, {equipment}. {part} needs changing, {failure} observed. Vendor on record: {supplier}. {priority}.",
    "Condition check {id} at {site}: {part} on {equipment} within limits, no {failure} found. {priority}.",
]
general_correspondence = [
    "Meeting summary: Discussed Q{quarter} procurement strategy for {category}. Action items include reviewing {supplier} contract terms.",
    "Internal memo: Budget allocation for {category} spare parts has been updated for Q{quarter}. Please review the revised figures.",
    "Team update: The new vendor onboarding process for {supplier} is now complete. They will supply {category} components starting next month.",
    "Policy reminder: All {category} purchases above USD 5000 require VP level approval. Please ensure compliance with procurement guidelines.",
    "Training notice: New procurement system training scheduled for next week. All {category} team members must attend. Contact HR for details.",
    "Memo: Stock of {part} at {site} will be reviewed in Q{quarter}. Please do not raise orders with {supplier} until the review is done.",
    "Agenda: Q{quarter} review of {category} spare parts, including {part} and {equipment}. Site visit planned for {site}.",
    "Reminder to all buyers: orders above 5000 USD for {part} need approval, and {supplier} delivery delays must be reported to management.",
    "Notes from supplier call with {supplier}: discussed {category} lead times and the delivery schedule for {site}.",
]

FIELDS = {"{part}": "PART", "{supplier}": "SUPPLIER", "{qty}": "QUANTITY",
          "{date}": "DATE", "{site}": "SITE", "{equipment}": "EQUIPMENT"}

def build(templates, doc_type, doc_id, **extra):
    """Fill one random template and record the gold (correct) entities."""
    tpl_idx = int(np.random.randint(len(templates)))
    template = templates[tpl_idx]
    vals = dict(
        id=np.random.randint(1000, 9999), qty=np.random.randint(5, 500),
        part=np.random.choice(parts), supplier=np.random.choice(suppliers),
        date=f"{np.random.choice([2024, 2025])}-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}",
        site=np.random.choice(sites), equipment=np.random.choice(equipment_list),
        failure=np.random.choice(failures), category=np.random.choice(categories),
        quarter=np.random.randint(1, 5), priority=np.random.choice(priority_tags),
        urgency="", status="", detail="",
    )
    vals.update(extra)
    gold = {ent: [] for ent in FIELDS.values()}
    for placeholder, ent in FIELDS.items():
        if placeholder in template:
            gold[ent].append(str(vals[placeholder.strip("{}")]).lower())
    return {"text": template.format(**vals), "doc_type": doc_type, "doc_id": doc_id,
            "gold": gold, "template_id": tpl_idx}

records = []
for i in range(150):                                   # Purchase Orders
    tag, lvl = urgency_tags[np.random.randint(len(urgency_tags))]
    r = build(purchase_orders, "Purchase Order", f"PO-{i+1:04d}", urgency=tag)
    r["gold_urgency"] = lvl
    records.append(r)

for i in range(150):                                   # Supplier Communications
    sentiment = np.random.choice(["positive", "negative", "neutral"], p=[0.3, 0.4, 0.3])
    if sentiment == "positive":
        status, detail = np.random.choice(statuses_positive), np.random.choice(details_positive)
    elif sentiment == "negative":
        status, detail = np.random.choice(statuses_negative), np.random.choice(details_negative)
    else:
        status, detail = np.random.choice(statuses_neutral), np.random.choice(details_neutral)
    # ~15% HARD cases: reassuring status but a problem in the detail (label = negative)
    if np.random.rand() < 0.15:
        status, detail, sentiment = (np.random.choice(statuses_positive),
                                     np.random.choice(details_negative), "negative")
    r = build(supplier_emails, "Supplier Communication", f"SC-{i+1:04d}",
              status=status, detail=detail)
    r["true_sentiment"] = sentiment
    records.append(r)

for i in range(150):                                   # Maintenance Reports
    records.append(build(maintenance_reports, "Maintenance Report", f"MR-{i+1:04d}"))
for i in range(100):                                   # General Correspondence
    records.append(build(general_correspondence, "General Correspondence", f"GC-{i+1:04d}"))

df = pd.DataFrame(records)

# OCR-style noise on ~25% of documents (the clean original is kept)
def add_ocr_noise(text, p=0.015):
    swaps = {"o": "0", "l": "1", "i": "1", "s": "5"}
    out = []
    for ch in text:
        r = np.random.rand()
        if ch.lower() in swaps and r < p:
            out.append(swaps[ch.lower()])
        elif r > 1 - p / 2:
            continue
        else:
            out.append(ch)
    return "".join(out)

df["text_original"] = df["text"]
df["is_noisy"] = np.random.rand(len(df)) < 0.25
df.loc[df["is_noisy"], "text"] = df.loc[df["is_noisy"], "text"].apply(add_ocr_noise)

# shuffle so document types are mixed
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Generated {len(df)} synthetic procurement documents "
      f"({int(df['is_noisy'].sum())} with OCR noise)\n")
print(df["doc_type"].value_counts().to_string())
print("\nSample (clean):", df.iloc[0]["text_original"])
print("Sample (as stored):", df.iloc[0]["text"])

Generated 550 synthetic procurement documents (137 with OCR noise)

doc_type
Supplier Communication    150
Purchase Order            150
Maintenance Report        150
General Correspondence    100

Sample (clean): Delivery advice from MechaComponents Ltd: 421x gearbox GB 300 for Jebel Ali Free Zone is under quality inspection. No issues reported at this stage. Please confirm receipt.
Sample (as stored): Delivery advice from MechaComponents Ltd: 421x gearbox GB 300 for Jebel Ali Free Zone is under quality inspection. No issues reported at this stage. Please confirm receipt.


## 3. Text Preprocessing Pipeline

The preprocessing pipeline performs three key operations:

1. **Text Cleaning**: Converts to lowercase, removes special characters,
   and normalises whitespace
2. **Abbreviation Expansion**: Expands domain specific abbreviations
   (e.g., PO → purchase order, MRO → maintenance repair and operations)
3. **Stopword Removal**: Removes common English stopwords while preserving
   domain specific terms that carry procurement meaning

In [4]:
# Abbreviation map for procurement domain
abbreviation_map = {
    "po": "purchase order", "wo": "work order",
    "mr": "maintenance report", "sr": "service report",
    "ir": "inspection report", "tn": "technician note",
    "qty": "quantity", "mro": "maintenance repair and operations",
    "eol": "end of life", "oem": "original equipment manufacturer",
    "eta": "estimated time of arrival", "moq": "minimum order quantity",
    "bom": "bill of materials",
    "hvac": "heating ventilation and air conditioning",
    "cnc": "computer numerical control"
}

# Common English stopwords
common_stopwords = {
    'a', 'an', 'the', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'will',
    'would', 'could', 'should', 'may', 'might', 'shall', 'can', 'need',
    'to', 'of', 'in', 'for', 'on', 'with', 'at', 'by', 'from', 'as',
    'into', 'through', 'during', 'before', 'after', 'above', 'below',
    'between', 'out', 'off', 'over', 'under', 'again', 'further', 'then',
    'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'each',
    'every', 'both', 'few', 'more', 'most', 'other', 'some', 'such', 'no',
    'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very',
    'just', 'because', 'but', 'and', 'or', 'if', 'while', 'about', 'up',
    'its', 'it', 'this', 'that', 'these', 'those', 'i', 'me', 'my',
    'we', 'our', 'you', 'your', 'he', 'him', 'his', 'she', 'her', 'they',
    'them', 'their', 'what', 'which', 'who', 'whom', 'am'
}

def clean_text(text):
    """Cleans procurement text by removing special characters and normalising."""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^\w\s\.\,\:\;\']', '', text)
    return text

def expand_abbreviations(text):
    """Expands domain specific abbreviations in procurement text."""
    words = text.split()
    return ' '.join([
        abbreviation_map.get(re.sub(r'[^\w]', '', w), w) for w in words
    ])

def remove_stopwords(text):
    """Removes common English stopwords while preserving domain terms."""
    return ' '.join([w for w in text.split() if w not in common_stopwords])

def preprocess_document(text):
    """Full preprocessing pipeline for procurement documents."""
    return remove_stopwords(expand_abbreviations(clean_text(text)))

# Apply preprocessing
df['cleaned_text'] = df['text'].apply(preprocess_document)

# Display before and after
print("ORIGINAL:")
print(f"  {df.iloc[0]['text']}\n")
print("PREPROCESSED:")
print(f"  {df.iloc[0]['cleaned_text']}")

ORIGINAL:
  Delivery advice from MechaComponents Ltd: 421x gearbox GB 300 for Jebel Ali Free Zone is under quality inspection. No issues reported at this stage. Please confirm receipt.

PREPROCESSED:
  delivery advice mechacomponents ltd: 421x gearbox gb 300 jebel ali free zone quality inspection. issues reported stage. please confirm receipt.


## 4. Module 1: Document Classification

**Goal:** assign each document to one of four types.

**Approach:** a scikit-learn `Pipeline` of `TfidfVectorizer` (1–2 grams, max 5,000 features) followed by a classifier. The vectoriser sits **inside** the pipeline so it is fitted only on training data, which avoids leakage.

**Five models are compared** (Logistic Regression, Linear SVM, Naive Bayes, Random Forest, Gradient Boosting) under three protocols:

1. **5-fold cross-validation:** the standard check.
2. **Unseen templates:** train on templates 0–6, test on templates 7–8. This is the honest test, because the model has never seen those sentence structures.
3. **Noisy documents:** performance on the OCR-noised subset.

The best model by unseen-template accuracy is used in the final demo. Expect near-perfect cross-validation scores (templates make the task easy) and a visible drop on unseen templates; that gap is the real finding.

In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_text = df["cleaned_text"]
y = df["doc_type"]

def make_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
        "Linear SVM":          LinearSVC(random_state=SEED),
        "Naive Bayes":         MultinomialNB(),
        "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=SEED),
        "Gradient Boosting":   GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=SEED),
    }

def make_pipe(model):
    # TF-IDF is fitted INSIDE the pipeline, so it only ever sees training data
    return make_pipeline(
        TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95),
        model)

# three test protocols
unseen = df["template_id"] >= 7          # templates 7-8 never seen in training
noisy  = df["is_noisy"]                  # train on clean text, test on OCR-noisy text
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

results = {}
for name in make_models():
    cv_scores = cross_val_score(make_pipe(make_models()[name]), X_text, y, cv=skf)
    p1 = make_pipe(make_models()[name]).fit(X_text[~unseen], y[~unseen])
    acc_unseen = accuracy_score(y[unseen], p1.predict(X_text[unseen]))
    p2 = make_pipe(make_models()[name]).fit(X_text[~noisy], y[~noisy])
    acc_noisy = accuracy_score(y[noisy], p2.predict(X_text[noisy]))
    results[name] = dict(cv_mean=cv_scores.mean(), cv_std=cv_scores.std(),
                         unseen=acc_unseen, noisy=acc_noisy,
                         unseen_pred=p1.predict(X_text[unseen]))

print(f"Documents: {len(df)} | unseen-template test: {unseen.sum()} | noisy test: {noisy.sum()}\n")
print(f"{'Model':<22}{'5-fold CV':>16}{'Unseen templates':>20}{'Noisy docs':>13}")
print("-" * 71)
for n, r in results.items():
    print(f"{n:<22}{r['cv_mean']*100:>9.1f}% ±{r['cv_std']*100:>4.1f}"
          f"{r['unseen']*100:>18.1f}%{r['noisy']*100:>12.1f}%")

best_name = max(results, key=lambda n: results[n]["unseen"])
print(f"\nBest on unseen templates (the hardest, most realistic test): {best_name}")
print(f"\nClassification report - {best_name} on unseen templates:\n")
print(classification_report(y[unseen], results[best_name]["unseen_pred"]))
labels = sorted(y.unique())
cm = confusion_matrix(y[unseen], results[best_name]["unseen_pred"], labels=labels)
print("Confusion matrix (rows = true, columns = predicted):")
print(pd.DataFrame(cm, index=[l[:12] for l in labels],
                   columns=[l[:8] for l in labels]).to_string())

# keep the variables the dashboard cell still uses (standard random 80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=SEED, stratify=y)
rf_pipe = make_pipe(RandomForestClassifier(n_estimators=100, random_state=SEED)).fit(X_train, y_train)
rf_pred = rf_pipe.predict(X_test)
print(f"\nFor reference - standard random 80/20 split (optimistic): "
      f"{accuracy_score(y_test, rf_pred)*100:.1f}%")

Documents: 550 | unseen-template test: 109 | noisy test: 137

Model                        5-fold CV    Unseen templates   Noisy docs
-----------------------------------------------------------------------
Logistic Regression       100.0% ± 0.0              86.2%       100.0%
Linear SVM                100.0% ± 0.0              91.7%       100.0%
Naive Bayes                99.6% ± 0.4              86.2%        99.3%
Random Forest             100.0% ± 0.0              82.6%       100.0%
Gradient Boosting          99.3% ± 0.7              56.0%        98.5%

Best on unseen templates (the hardest, most realistic test): Linear SVM

Classification report - Linear SVM on unseen templates:

                        precision    recall  f1-score   support

General Correspondence       1.00      0.56      0.71        18
    Maintenance Report       0.96      1.00      0.98        27
        Purchase Order       0.97      0.97      0.97        33
Supplier Communication       0.82      1.00      0.

## 5. Module 2: Named Entity Recognition (NER)

This module extracts structured information from procurement text,
identifying seven entity types:

| Entity Type | Examples |
|---|---|
| PART | bearing assembly 6205 2RS, hydraulic pump HP 200 |
| SUPPLIER | PartsPro International, TechSupply Global |
| QUANTITY | 275 units, 50 pieces |
| DATE | 2024-05-07, 2024-12-15 |
| URGENCY | CRITICAL, HIGH, ROUTINE |
| SITE | Dubai Logistics Hub, Jebel Ali Free Zone |
| EQUIPMENT | Conveyor System A1, HVAC System H4 |

The implementation uses rule based pattern matching with regular expressions.
In production, this would be replaced with a fine tuned BERT NER model
trained on annotated procurement data using BIO tagging.

In [6]:
# ============================================================
# MODULE 2: NER (rule-based) + EVALUATION against gold labels
# Self-contained: defines the extractor AND evaluates it.
# Needs: the generator cell (df, suppliers, sites, equipment_list)
# ============================================================
import re
import numpy as np
import pandas as pd

# ---- 0. Safety check: were the earlier cells run? ----
_required = ["df", "suppliers", "sites", "equipment_list"]
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise RuntimeError(
        f"Missing {_missing}. Run the data-generation cell first "
        "(Runtime > Run before, or Run all), then run this cell again.")
if "gold" not in df.columns or "text_original" not in df.columns:
    raise RuntimeError("df has no 'gold'/'text_original' columns - re-run the NEW generator cell.")

# ---- 1. Gazetteers and patterns ----
supplier_names  = [s.lower() for s in suppliers]
site_names      = [s.lower() for s in sites]
equipment_names = [e.lower() for e in equipment_list]

part_patterns = [
    r'\b(bearing assembly \w+ \w+)\b', r'\b(hydraulic pump \w+ \w+)\b',
    r'\b(conveyor belt \w+ \w+)\b',    r'\b(servo motor \w+ \w+)\b',
    r'\b(gearbox \w+ \w+)\b',          r'\b(control panel \w+ \w+)\b',
    r'\b(air filter \w+ \w+)\b',       r'\b(power supply unit \w+ \w+)\b',
    r'\b(sensor module \w+ \w+)\b',    r'\b(drive shaft \w+ \w+)\b',
    r'\b(cooling fan \w+ \w+)\b',      r'\b(pressure valve \w+ \w+)\b',
    r'\b(transformer \w+ \w+)\b',      r'\b(brake pad \w+ \w+)\b',
    r'\b(compressor unit \w+ \w+)\b',
]
part_words = (r'(?:bearing assembly|hydraulic pump|conveyor belt|servo motor|gearbox|'
              r'control panel|air filter|power supply unit|sensor module|drive shaft|'
              r'cooling fan|pressure valve|transformer|brake pad|compressor unit)')

urgency_keywords = {
    'critical': 'CRITICAL', 'urgent': 'HIGH', 'emergency': 'CRITICAL',
    'expedite': 'HIGH', 'immediate': 'CRITICAL', 'fast track': 'HIGH',
    'high priority': 'HIGH', 'routine': 'ROUTINE', 'standard': 'ROUTINE',
    'no rush': 'ROUTINE'
}

# ---- 2. Extractor ----
def extract_entities(text):
    """Extracts named entities from procurement text."""
    text_lower = text.lower()
    entities = {'PART': [], 'SUPPLIER': [], 'QUANTITY': [], 'DATE': [],
                'URGENCY': 'ROUTINE', 'SITE': [], 'EQUIPMENT': []}

    for pattern in part_patterns:
        entities['PART'].extend(re.findall(pattern, text_lower))
    for supplier in supplier_names:
        if supplier in text_lower:
            entities['SUPPLIER'].append(supplier.title())

    qty_m1 = re.findall(r'\b(\d+)\s*(?:units?|pieces?|pcs|x)\b', text_lower)   # "50 units", "421x"
    qty_m2 = re.findall(r'(?:qty|quantity)\s*[:\s]*(\d+)', text_lower)         # "qty: 145"
    qty_m3 = re.findall(r'\b(\d+)\s+' + part_words, text_lower)                # "275 gearbox GB 300"
    entities['QUANTITY'] = list(set(qty_m1 + qty_m2 + qty_m3))

    entities['DATE'] = re.findall(r'\b(\d{4}[\-/]\d{1,2}[\-/]\d{1,2})\b', text)

    for keyword, level in urgency_keywords.items():
        if keyword in text_lower:
            if level == 'CRITICAL':
                entities['URGENCY'] = 'CRITICAL'
                break
            elif level == 'HIGH' and entities['URGENCY'] != 'CRITICAL':
                entities['URGENCY'] = 'HIGH'

    for site in site_names:
        if site in text_lower:
            entities['SITE'].append(site.title())
    for equip in equipment_names:
        if equip in text_lower:
            entities['EQUIPMENT'].append(equip.title())
    return entities

# Apply to every document (used later by the RAG module)
df['entities'] = df['text'].apply(extract_entities)

# ---- 3. Evaluation ----
ENT_TYPES = ["PART", "SUPPLIER", "QUANTITY", "DATE", "SITE", "EQUIPMENT"]

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

def evaluate_ner(frame, text_col):
    counts = {e: [0, 0, 0] for e in ENT_TYPES}          # tp, fp, fn
    for _, row in frame.iterrows():
        pred = extract_entities(row[text_col])
        for e in ENT_TYPES:
            g = set(row["gold"][e])
            p = set(str(x).lower() for x in pred[e])
            counts[e][0] += len(g & p)
            counts[e][1] += len(p - g)
            counts[e][2] += len(g - p)
    return counts

noisy_docs = df[df["is_noisy"]]
c_clean = evaluate_ner(df, "text_original")       # clean text
c_noisy = evaluate_ner(noisy_docs, "text")        # OCR-noisy text

print(f"{'Entity':<11}{'Gold':>6} | {'Clean P':>8}{'R':>7}{'F1':>7} | {'Noisy P':>8}{'R':>7}{'F1':>7}")
print("-" * 66)
tot_c, tot_n = [0, 0, 0], [0, 0, 0]
for e in ENT_TYPES:
    pc, rc, fc = prf(*c_clean[e]); pn, rn, fn_ = prf(*c_noisy[e])
    gold_n = c_clean[e][0] + c_clean[e][2]
    print(f"{e:<11}{gold_n:>6} | {pc:>8.3f}{rc:>7.3f}{fc:>7.3f} | {pn:>8.3f}{rn:>7.3f}{fn_:>7.3f}")
    for i in range(3):
        tot_c[i] += c_clean[e][i]; tot_n[i] += c_noisy[e][i]
pc, rc, fc = prf(*tot_c); pn, rn, fn_ = prf(*tot_n)
print("-" * 66)
print(f"{'MICRO AVG':<11}{'':>6} | {pc:>8.3f}{rc:>7.3f}{fc:>7.3f} | {pn:>8.3f}{rn:>7.3f}{fn_:>7.3f}")

po = df[df["doc_type"] == "Purchase Order"]
u_clean = np.mean([extract_entities(t)["URGENCY"] == g for t, g in zip(po["text_original"], po["gold_urgency"])])
po_n = po[po["is_noisy"]]
u_noisy = np.mean([extract_entities(t)["URGENCY"] == g for t, g in zip(po_n["text"], po_n["gold_urgency"])])
print(f"\nUrgency accuracy on Purchase Orders: clean {u_clean*100:.1f}% | noisy {u_noisy*100:.1f}% (n={len(po_n)})")

print("\nExamples of entities missed on noisy documents:")
shown = 0
for _, row in noisy_docs.iterrows():
    pred = extract_entities(row["text"])
    for e in ENT_TYPES:
        missed = set(row["gold"][e]) - set(str(x).lower() for x in pred[e])
        if missed and shown < 6:
            print(f"  [{e}] missed {sorted(missed)}\n     text: {row['text'][:110]}...")
            shown += 1

Entity       Gold |  Clean P      R     F1 |  Noisy P      R     F1
------------------------------------------------------------------
PART          484 |    1.000  1.000  1.000 |    0.973  0.872  0.920
SUPPLIER      385 |    1.000  1.000  1.000 |    1.000  0.779  0.876
QUANTITY      232 |    1.000  1.000  1.000 |    0.984  0.940  0.962
DATE          150 |    1.000  1.000  1.000 |    1.000  0.974  0.987
SITE          251 |    1.000  1.000  1.000 |    1.000  0.782  0.878
EQUIPMENT     180 |    1.000  1.000  1.000 |    1.000  0.822  0.902
------------------------------------------------------------------
MICRO AVG         |    1.000  1.000  1.000 |    0.989  0.853  0.916

Urgency accuracy on Purchase Orders: clean 100.0% | noisy 100.0% (n=39)

Examples of entities missed on noisy documents:
  [SUPPLIER] missed ['partspro international']
     text: Notification from PartsPro Internat1onal: Your order for bearing assembly 6205 2RS is confirmed and on schedul...
  [SUPPLIER] missed ['partsp

## 6. Module 3: Sentiment and Urgency Analysis

This module classifies supplier communications along two dimensions:

- **Sentiment**: Positive, Negative, or Neutral
- **Urgency**: Critical, High, or Routine

The rule based approach uses keyword indicators specific to procurement
communications. In production, this would be replaced with fine tuned
BERT classifiers trained on labelled procurement email datasets.

In [7]:
positive_indicators = [
    'confirmed', 'on schedule', 'ahead of schedule', 'ready for dispatch',
    'passed quality', 'bonus', 'no extra cost', 'express shipping',
    'completed successfully', 'approved', 'fulfilled', 'excellent'
]

negative_indicators = [
    'delayed', 'shortage', 'on hold', 'backordered', 'defects',
    'rework', 'price increase', 'customs clearance issues', 'regret',
    'unfortunately', 'problem', 'failure', 'cancelled', 'rejected',
    'overdue', 'penalty', 'escalat'
]

neutral_indicators = [
    'being processed', 'under inspection', 'awaiting confirmation',
    'standard lead time', 'no issues reported', 'tracking details',
    'in progress', 'pending review'
]

def classify_sentiment(text):
    """Classifies document sentiment based on keyword indicators."""
    text_lower = text.lower()
    scores = {
        'positive': sum(1 for ind in positive_indicators if ind in text_lower),
        'negative': sum(1 for ind in negative_indicators if ind in text_lower),
        'neutral': sum(1 for ind in neutral_indicators if ind in text_lower)
    }
    total = sum(scores.values())
    if total == 0:
        return 'neutral', 0.5
    max_sent = max(scores, key=scores.get)
    return max_sent, round(scores[max_sent] / total, 2)

def classify_urgency(text):
    """Classifies document urgency level."""
    text_lower = text.lower()
    crit = sum(1 for w in ['critical', 'immediate', 'emergency', 'halted', 'safety']
               if w in text_lower)
    high = sum(1 for w in ['urgent', 'expedite', 'fast track', 'high priority', 'downtime']
               if w in text_lower)
    rout = sum(1 for w in ['routine', 'standard', 'no rush', 'scheduled', 'planned']
               if w in text_lower)
    total = crit + high + rout
    if total == 0:
        return 'ROUTINE', 0.5
    if crit > 0 and crit >= high:
        return 'CRITICAL', round(crit / total, 2)
    elif high > 0 and high > rout:
        return 'HIGH', round(high / total, 2)
    return 'ROUTINE', round(rout / total, 2)

# Apply to supplier communications
supplier_docs = df[df['doc_type'] == 'Supplier Communication'].copy()
supplier_docs[['predicted_sentiment', 'sentiment_confidence']] = (
    supplier_docs['text'].apply(lambda x: pd.Series(classify_sentiment(x)))
)
supplier_docs[['predicted_urgency', 'urgency_confidence']] = (
    supplier_docs['text'].apply(lambda x: pd.Series(classify_urgency(x)))
)

# Evaluate accuracy
correct = (supplier_docs['predicted_sentiment'] == supplier_docs['true_sentiment']).sum()
accuracy = correct / len(supplier_docs) * 100

print(f"Sentiment Classification Accuracy: {accuracy:.1f}%")
print(f"Average Confidence Score: {supplier_docs['sentiment_confidence'].mean():.2f}\n")

print("Sentiment Distribution:")
for sent, count in supplier_docs['predicted_sentiment'].value_counts().items():
    print(f"  {sent.capitalize():12s}: {count} ({count/len(supplier_docs)*100:.1f}%)")

print(f"\nConfusion Matrix:")
print(f"{'':15s} {'Pred Pos':>10s} {'Pred Neg':>10s} {'Pred Neu':>10s}")
for true_s in ['positive', 'negative', 'neutral']:
    row = supplier_docs[supplier_docs['true_sentiment'] == true_s]
    pc = row['predicted_sentiment'].value_counts()
    print(f"  True {true_s.capitalize():9s}: "
          f"{pc.get('positive',0):>8d} {pc.get('negative',0):>8d} {pc.get('neutral',0):>8d}")

# Examples
print("\nNegative Example:")
neg = supplier_docs[supplier_docs['predicted_sentiment'] == 'negative'].iloc[0]
print(f"  {neg['text'][:150]}...")
print(f"  Sentiment: {neg['predicted_sentiment']} | Confidence: {neg['sentiment_confidence']}")

Sentiment Classification Accuracy: 86.0%
Average Confidence Score: 0.91

Sentiment Distribution:
  Negative    : 64 (42.7%)
  Neutral     : 45 (30.0%)
  Positive    : 41 (27.3%)

Confusion Matrix:
                  Pred Pos   Pred Neg   Pred Neu
  True Positive :       27        0        0
  True Negative :       14       57        0
  True Neutral  :        0        7       45

Negative Example:
  Hi team, regarding PO1475 (9 pieces of bearing assembly 6205 2RS) delivering to Abu Dhabi Warehouse: status is delayed due to supply shortage. Raw mat...
  Sentiment: negative | Confidence: 1.0


## 7. Module 4: Retrieval-Augmented Generation (RAG)

**Goal:** let a user ask a natural-language question and get an answer grounded in the documents.

**Step 5A, Retrieval.** Three retrievers are built over the corpus:
- **Sparse:** TF-IDF cosine similarity (exact keyword match).
- **Dense:** `all-MiniLM-L6-v2` embeddings computed on the **GPU**, indexed with FAISS (inner product on normalised vectors).
- **Hybrid:** Reciprocal Rank Fusion (k = 60) of the sparse and dense rankings.

**Step 5B, Evaluation.** 30 queries, with relevance defined from the gold labels, scored by Hit@1, Hit@3, MRR and Precision@5, overall and by document type. This shows which retriever suits which query type, rather than assuming one is best.

**Step 6, Grounded generation.** `Qwen2.5-3B-Instruct` (fp16, GPU) answers using only the retrieved passages and must cite them. Three safeguards control hallucination:
1. A **refusal gate:** if the best dense similarity is below a calibrated threshold, the system answers "Information not found in the available documents."
2. A **grounding check:** entities in the answer must appear in the cited sources.
3. **Citation checks:** citations must point to retrieved documents.

**Step 6B, Safety tests.** Off-topic questions (e.g. sports, weather) and in-domain unanswerable questions are used on purpose to check that the system refuses instead of inventing answers.

In [8]:
# ============================================================
# MODULE 4 (RAG) - PART A: retrieval indexes
#   sparse  = TF-IDF (baseline)
#   dense   = sentence-transformer embeddings on the GPU + FAISS
#   hybrid  = reciprocal rank fusion of both
# Needs: df (with 'entities' from the NER cell), SEED, imports from earlier cells
# ============================================================
import time
import torch
import faiss
from collections import Counter
from sentence_transformers import SentenceTransformer

if "entities" not in df.columns:
    raise RuntimeError("Run the NER cell first (it creates df['entities']).")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU - check Runtime > Change runtime type)")

# ---- enriched text for the sparse index (as before) ----
def create_enriched_text(row):
    enriched = row["text"]
    ent = row["entities"]
    if ent["PART"]:     enriched += " Parts: " + ", ".join(ent["PART"])
    if ent["SUPPLIER"]: enriched += " Suppliers: " + ", ".join(ent["SUPPLIER"])
    if ent["SITE"]:     enriched += " Sites: " + ", ".join(ent["SITE"])
    enriched += f" Urgency: {ent['URGENCY']}"
    return enriched

df["enriched_text"] = df.apply(create_enriched_text, axis=1)

# ---- 1. sparse index ----
rag_tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
doc_vectors = rag_tfidf.fit_transform(df["enriched_text"])

# ---- 2. dense index: embeddings computed on the GPU, stored in FAISS ----
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)
t0 = time.time()
doc_emb = emb_model.encode(df["text"].tolist(), batch_size=64, normalize_embeddings=True,
                           show_progress_bar=False, convert_to_numpy=True).astype("float32")
faiss_index = faiss.IndexFlatIP(doc_emb.shape[1])      # inner product = cosine (vectors normalised)
faiss_index.add(doc_emb)
print(f"Dense index: {faiss_index.ntotal} docs x {doc_emb.shape[1]} dims "
      f"({EMB_MODEL}, encoded in {time.time()-t0:.1f}s on {DEVICE})")
print(f"Sparse index: {doc_vectors.shape[0]} docs x {doc_vectors.shape[1]} features")

# ---- 3. search functions (return [(row_position, score), ...]) ----
def search_tfidf(q, k=10):
    sims = cosine_similarity(rag_tfidf.transform([q.lower()]), doc_vectors).ravel()
    idx = sims.argsort()[::-1][:k]
    return [(int(i), float(sims[i])) for i in idx]

def search_dense(q, k=10):
    qv = emb_model.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    s, i = faiss_index.search(qv, k)
    return [(int(a), float(b)) for a, b in zip(i[0], s[0])]

def search_hybrid(q, k=10, rrf_k=60):
    """Reciprocal rank fusion: documents ranked highly by BOTH retrievers win."""
    scores = {}
    for lst in (search_tfidf(q, 50), search_dense(q, 50)):
        for rank, (i, _) in enumerate(lst):
            scores[i] = scores.get(i, 0.0) + 1.0 / (rrf_k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])[:k]

SEARCHERS = {"TF-IDF (sparse)": search_tfidf, "Dense (GPU + FAISS)": search_dense, "Hybrid (RRF)": search_hybrid}

def retrieve_documents(query, top_k=3, method="Hybrid (RRF)"):
    hits = SEARCHERS[method](query, top_k)
    return [{"doc_id": df.iloc[i]["doc_id"], "doc_type": df.iloc[i]["doc_type"],
             "text": df.iloc[i]["text"], "entities": df.iloc[i]["entities"],
             "similarity": round(s, 4)} for i, s in hits]

def generate_response(query, docs):
    """Template answer built only from retrieved documents (LLM generation comes in the next step)."""
    parts_found, supps_found, sites_found = [], [], []
    for d in docs:
        parts_found.extend(d["entities"].get("PART", []))
        supps_found.extend(d["entities"].get("SUPPLIER", []))
        sites_found.extend(d["entities"].get("SITE", []))
    response = [f"Query: {query}", f"Retrieved {len(docs)} relevant documents:\n"]
    if parts_found:
        top_part = Counter(parts_found).most_common(1)[0]
        response.append(f"  Most referenced part: {top_part[0].title()} ({top_part[1]} mentions)")
    if supps_found:
        response.append(f"  Suppliers: {', '.join(sorted(set(supps_found)))}")
    if sites_found:
        response.append(f"  Sites: {', '.join(sorted(set(sites_found)))}")
    response.append("\n  Sources:")
    for i, d in enumerate(docs):
        response.append(f"    [{i+1}] {d['doc_type']} ({d['doc_id']}) | score: {d['similarity']}")
        response.append(f"        {d['text'][:120]}...")
    return "\n".join(response)

# ---- 4. quick demo ----
for q in ["What is the status of bearing assembly orders?",
          "Are there any delays from suppliers affecting hydraulic pumps?",
          "What equipment failures have been reported at Dubai Logistics Hub?"]:
    print("=" * 60)
    print(generate_response(q, retrieve_documents(q)))

Embedding device: cuda Tesla T4


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dense index: 550 docs x 384 dims (sentence-transformers/all-MiniLM-L6-v2, encoded in 0.7s on cuda)
Sparse index: 550 docs x 3000 features
Query: What is the status of bearing assembly orders?
Retrieved 3 relevant documents:

  Most referenced part: Bearing Assembly 6205 2Rs (3 mentions)
  Suppliers: Globalmech Industries, Partspro International, Sparemax Solutions
  Sites: Dubai Logistics Hub

  Sources:
    [1] Supplier Communication (SC-0110) | score: 0.0318
        Subject: Order Update. The delivery of bearing assembly 6205 2RS from PartsPro International is backordered until next m...
    [2] Supplier Communication (SC-0001) | score: 0.0313
        Subject: Order Update. The delivery of bearing assembly 6205 2RS from GlobalMech Industries is delayed due to supply sho...
    [3] General Correspondence (GC-0025) | score: 0.0305
        Memo: Stock of bearing assembly 6205 2RS at Dubai Logistics Hub will be reviewed in Q3. Please do not raise orders with ...
Query: Are there any dela

In [9]:
# ============================================================
# MODULE 4 (RAG) - PART B: retrieval evaluation (30 queries)
# Relevance is defined automatically from the gold labels, so no hand-labelling.
# ============================================================
rng = np.random.RandomState(SEED)

def relevant_ids(doc_type, part=None, supplier=None, site=None, equipment=None, sentiment=None):
    ids = set()
    for _, r in df.iterrows():
        if r["doc_type"] != doc_type:
            continue
        g = r["gold"]
        if part and part.lower() not in g["PART"]: continue
        if supplier and supplier.lower() not in g["SUPPLIER"]: continue
        if site and site.lower() not in g["SITE"]: continue
        if equipment and equipment.lower() not in g["EQUIPMENT"]: continue
        if sentiment and r.get("true_sentiment") != sentiment: continue
        ids.add(r["doc_id"])
    return ids

# queries are phrased DIFFERENTLY from the document templates (tests semantic matching)
po_q = ["Which purchase orders were raised for {part} with {supplier}?",
        "Find orders where {supplier} is the vendor of {part}",
        "Show requisitions for {part} to be sourced from {supplier}"]
neg_q = ["Which suppliers reported delays or problems with {part}?",
         "Any supplier updates warning of a hold-up on {part}?",
         "Which vendors told us {part} is running late or has quality issues?"]
pos_q = ["Which suppliers confirmed {part} is on schedule or ready to ship?",
         "Any good news from vendors about {part} deliveries?"]
mr_q  = ["What maintenance work was done on {equipment} at {site}?",
         "Breakdown or repair records for {equipment} in {site}",
         "Which parts failed on {equipment} at {site}?"]

eval_set = []
def sample_docs(doc_type, n):
    return df[df["doc_type"] == doc_type].sample(n=n, random_state=SEED)

for k, (_, r) in enumerate(sample_docs("Purchase Order", 10).iterrows()):
    part, sup = r["gold"]["PART"][0], r["gold"]["SUPPLIER"][0]
    eval_set.append(("Purchase Order", po_q[k % 3].format(part=part, supplier=sup.title()),
                     relevant_ids("Purchase Order", part=part, supplier=sup)))
for k, (_, r) in enumerate(sample_docs("Supplier Communication", 10).iterrows()):
    part, sent = r["gold"]["PART"][0], r["true_sentiment"]
    if sent == "negative":
        q = neg_q[k % 3].format(part=part)
    elif sent == "positive":
        q = pos_q[k % 2].format(part=part)
    else:
        sent = "negative"; q = neg_q[k % 3].format(part=part)       # neutral -> ask the negative question
    eval_set.append(("Supplier Communication", q, relevant_ids("Supplier Communication", part=part, sentiment=sent)))
for k, (_, r) in enumerate(sample_docs("Maintenance Report", 10).iterrows()):
    eq, st = r["gold"]["EQUIPMENT"][0], r["gold"]["SITE"][0]
    eval_set.append(("Maintenance Report", mr_q[k % 3].format(equipment=eq.title(), site=st.title()),
                     relevant_ids("Maintenance Report", equipment=eq, site=st)))
eval_set = [e for e in eval_set if len(e[2]) > 0]
print(f"Evaluation set: {len(eval_set)} queries | avg relevant docs per query: "
      f"{np.mean([len(e[2]) for e in eval_set]):.1f}\n")

def score_method(search_fn):
    rows = []
    for qtype, q, rel in eval_set:
        ranked = [df.iloc[i]["doc_id"] for i, _ in search_fn(q, 10)]
        first = next((rank + 1 for rank, d in enumerate(ranked) if d in rel), None)
        rows.append({"type": qtype,
                     "hit1": float(first == 1),
                     "hit3": float(first is not None and first <= 3),
                     "mrr": 1.0 / first if first else 0.0,
                     "p5": sum(d in rel for d in ranked[:5]) / 5})
    return pd.DataFrame(rows)

all_scores = {name: score_method(fn) for name, fn in SEARCHERS.items()}
print(f"{'Method':<22}{'Hit@1':>8}{'Hit@3':>8}{'MRR':>8}{'P@5':>8}")
print("-" * 54)
for name, s in all_scores.items():
    print(f"{name:<22}{s.hit1.mean():>8.3f}{s.hit3.mean():>8.3f}{s.mrr.mean():>8.3f}{s.p5.mean():>8.3f}")

print("\nMRR by query type:")
by_type = pd.DataFrame({n: s.groupby("type")["mrr"].mean() for n, s in all_scores.items()})
print(by_type.round(3).to_string())

Evaluation set: 30 queries | avg relevant docs per query: 4.1

Method                   Hit@1   Hit@3     MRR     P@5
------------------------------------------------------
TF-IDF (sparse)          0.600   0.667   0.662   0.433
Dense (GPU + FAISS)      0.433   0.800   0.597   0.340
Hybrid (RRF)             0.567   0.767   0.681   0.440

MRR by query type:
                        TF-IDF (sparse)  Dense (GPU + FAISS)  Hybrid (RRF)
type                                                                      
Maintenance Report                1.000                0.750         0.950
Purchase Order                    0.812                0.308         0.703
Supplier Communication            0.173                0.733         0.388


In [10]:
# ============================================================
# MODULE 4 (RAG) - PART C: grounded answer generation with an open LLM on the GPU
# Needs: retrieve_documents / search_dense (Part A), extract_entities (NER cell), eval_set (Part B)
# ============================================================
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Change this one line to try another model (e.g. "tiiuae/Falcon3-3B-Instruct", the UAE TII model)
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
try:
    llm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")
except TypeError:                                   # older transformers versions
    llm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
llm.eval()
print(f"Loaded {MODEL_NAME} on {llm.device} | GPU memory used: "
      f"{torch.cuda.memory_allocated()/1e9:.1f} GB" if torch.cuda.is_available() else f"Loaded {MODEL_NAME} on CPU (slow!)")

SYSTEM_PROMPT = (
    "You are a procurement assistant. Answer ONLY from the numbered documents provided. "
    "Cite the document numbers you used, like [1] or [2]. "
    "If the documents do not contain the answer, reply exactly: "
    "'Information not found in the available documents.' Do not invent facts.")

NOT_FOUND = "Information not found in the available documents."
MIN_DENSE_SIM = 0.25      # retrieval-confidence gate (calibrate it with the off-topic queries below)

def build_prompt(query, docs):
    context = "\n".join(f"[{i+1}] ({d['doc_id']}, {d['doc_type']}) {d['text']}" for i, d in enumerate(docs))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {query}\n"
                                    "Answer in 2-3 sentences and cite the documents."},
    ]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def rag_answer(query, top_k=5, method="Hybrid (RRF)", max_new_tokens=140):
    """Retrieve -> confidence gate -> LLM answer grounded in the retrieved documents."""
    best_sim = search_dense(query, 1)[0][1]                # cosine similarity of the best dense match
    docs = retrieve_documents(query, top_k=top_k, method=method)
    if best_sim < MIN_DENSE_SIM:                           # nothing relevant: refuse instead of guessing
        return {"answer": NOT_FOUND, "docs": docs, "best_sim": best_sim, "gated": True}
    inputs = tok(build_prompt(query, docs), return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return {"answer": answer, "docs": docs, "best_sim": best_sim, "gated": False}

def grounding_check(answer, docs):
    """Reuse the NER: every part/supplier/site/equipment/quantity in the answer must appear in the documents."""
    context = " ".join(d["text"] for d in docs).lower()
    ents = extract_entities(answer)
    claims = []
    for key in ["PART", "SUPPLIER", "SITE", "EQUIPMENT", "QUANTITY", "DATE"]:
        claims += [str(x).lower() for x in ents[key]]
    grounded = sum(c in context for c in claims)
    return grounded, len(claims)

# In-domain but UNANSWERABLE questions (sound like procurement, not in the data)
unanswerable = [
    "What is the price of the turbine blade TB 999?",
    "Which supplier delivered servo motors to the Riyadh warehouse?",
    "What was the 2023 budget for hydraulic hoses?",
    "How many employees work at PartsPro International?",
    "Which parts were ordered from Siemens Middle East?",
]
un_refused = [rag_answer(q)["answer"].strip().startswith("Information not found") for q in unanswerable]
print(f"\nIn-domain but unanswerable questions correctly refused: {np.mean(un_refused)*100:.1f}% (n={len(unanswerable)})")
for q, ref in zip(unanswerable, un_refused):
    print("  ", "REFUSED " if ref else "ANSWERED", "|", q)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-3B-Instruct on cuda:0 | GPU memory used: 6.3 GB

In-domain but unanswerable questions correctly refused: 100.0% (n=5)
   REFUSED  | What is the price of the turbine blade TB 999?
   REFUSED  | Which supplier delivered servo motors to the Riyadh warehouse?
   REFUSED  | What was the 2023 budget for hydraulic hoses?
   REFUSED  | How many employees work at PartsPro International?
   REFUSED  | Which parts were ordered from Siemens Middle East?


In [11]:
# ============================================================
# MODULE 4 (RAG) - PART D: calibrate the refusal gate + evaluate generated answers
# Needs: Part A-C cells already run (rag_answer, grounding_check, eval_set, search_dense)
# ============================================================
import re

off_topic = [
    "What is the weather forecast for Dubai tomorrow?",
    "Who won the football world cup?",
    "How do I bake a chocolate cake?",
    "What is the capital of France?",
    "What is the price of Bitcoin today?",
    "Recommend a good restaurant in Dubai",
    "How many people live in Dubai?",
    "Explain quantum computing in simple terms",
    "Who is the CEO of Tesla?",
    "What are the opening hours of the Dubai Mall?",
]
in_scope = [(qtype, q, rel) for qtype, q, rel in eval_set][::2]       # every 2nd query -> about 15

# In-domain but UNANSWERABLE questions (sound like procurement, not in the data)
unanswerable = [
    "What is the price of the turbine blade TB 999?",
    "Which supplier delivered servo motors to the Riyadh warehouse?",
    "What was the 2023 budget for hydraulic hoses?",
    "How many employees work at PartsPro International?",
    "Which parts were ordered from Siemens Middle East?",
]

# ---- 1. calibrate the gate from data ----
sims_in  = [search_dense(q, 1)[0][1] for _, q, _ in in_scope]
sims_off = [search_dense(q, 1)[0][1] for q in off_topic]
print(f"Best dense similarity | in-scope: min {min(sims_in):.2f}, mean {np.mean(sims_in):.2f}"
      f" | off-topic: max {max(sims_off):.2f}, mean {np.mean(sims_off):.2f}")
if max(sims_off) < min(sims_in):
    MIN_DENSE_SIM = round((max(sims_off) + min(sims_in)) / 2, 2)
    print(f"Classes are separable -> gate set to MIN_DENSE_SIM = {MIN_DENSE_SIM}")
else:
    print(f"Classes overlap -> keeping MIN_DENSE_SIM = {MIN_DENSE_SIM}; the LLM prompt does the rest")

# ---- 2. evaluate generated answers ----
def cited_indices(answer, n_docs):
    return sorted({int(x) for x in re.findall(r"\[(\d+)\]", answer) if 1 <= int(x) <= n_docs})

rows = []
for qtype, q, rel in in_scope:
    r = rag_answer(q)
    refused = r["answer"].strip().startswith("Information not found")
    g, n = grounding_check(r["answer"], r["docs"])
    cited = cited_indices(r["answer"], len(r["docs"]))
    cited_rel = [i for i in cited if r["docs"][i - 1]["doc_id"] in rel]
    rows.append({"type": qtype, "refused": refused,
                 "grounded": g / n if n else np.nan,
                 "has_citation": len(cited) > 0,
                 "cite_precision": len(cited_rel) / len(cited) if cited else np.nan})
res = pd.DataFrame(rows)

off_refused = [rag_answer(q)["answer"].strip().startswith("Information not found") for q in off_topic]
un_refused  = [rag_answer(q)["answer"].strip().startswith("Information not found") for q in unanswerable]

print(f"\nIn-scope queries evaluated: {len(res)}")
print(f"  Wrongly refused:               {res.refused.mean()*100:5.1f}%")
print(f"  Entity grounding rate:         {np.nanmean(res.grounded)*100:5.1f}%")
print(f"  Answers with a citation:       {res.has_citation.mean()*100:5.1f}%")
print(f"  Citation precision (relevant): {np.nanmean(res.cite_precision)*100:5.1f}%")
print(f"Off-topic queries evaluated: {len(off_topic)}")
print(f"  Correctly refused:             {np.mean(off_refused)*100:5.1f}%")
print(f"In-domain but unanswerable questions correctly refused: "
      f"{np.mean(un_refused)*100:.1f}% (n={len(unanswerable)})")
for q, ref in zip(unanswerable, un_refused):
    print("  ", "REFUSED " if ref else "ANSWERED", "|", q)
print("\nBy query type (citation precision):")
print(res.groupby("type")["cite_precision"].mean().round(3).to_string())

Best dense similarity | in-scope: min 0.68, mean 0.78 | off-topic: max 0.37, mean 0.24
Classes are separable -> gate set to MIN_DENSE_SIM = 0.53

In-scope queries evaluated: 15
  Wrongly refused:                 0.0%
  Entity grounding rate:         100.0%
  Answers with a citation:       100.0%
  Citation precision (relevant):  86.7%
Off-topic queries evaluated: 10
  Correctly refused:             100.0%
In-domain but unanswerable questions correctly refused: 100.0% (n=5)
   REFUSED  | What is the price of the turbine blade TB 999?
   REFUSED  | Which supplier delivered servo motors to the Riyadh warehouse?
   REFUSED  | What was the 2023 budget for hydraulic hoses?
   REFUSED  | How many employees work at PartsPro International?
   REFUSED  | Which parts were ordered from Siemens Middle East?

By query type (citation precision):
type
Maintenance Report        0.933
Purchase Order            0.767
Supplier Communication    0.900


In [14]:
# ============================================================
# STEP 7: END-TO-END PIPELINE + DEMO
#   process_document(text)  -> type, entities, urgency, sentiment, similar documents
#   ask(question)           -> grounded LLM answer with sources
#   Demo tabs: (1) analyse one email/document, (2) batch upload, (3) ask the documents
# Needs: preprocessing, classification, NER, sentiment and RAG cells already run
# ============================================================
import os, io, email, json
from email import policy
from sklearn.svm import LinearSVC

# 1. final classifier: the best model from Step 3, trained on all documents
final_clf = make_pipe(LinearSVC(random_state=SEED)).fit(df["cleaned_text"], df["doc_type"])

def process_document(text, n_similar=3):
    """Full pipeline for ONE raw procurement document."""
    cleaned = preprocess_document(text)
    scores = final_clf.decision_function([cleaned])[0]
    order = np.argsort(scores)[::-1]
    classes = final_clf.classes_
    doc_type = classes[order[0]]
    margin = float(scores[order[0]] - scores[order[1]])          # how far ahead of the runner-up

    ents = extract_entities(text)
    entities = {k: v for k, v in ents.items() if v and k != "URGENCY"}

    result = {
        "document_type": doc_type,
        "classifier_margin": round(margin, 2),
        "entities": entities,
        "urgency": ents["URGENCY"],
    }
    if doc_type == "Supplier Communication":                      # sentiment only makes sense for supplier emails
        label, conf = classify_sentiment(text)
        result["sentiment"] = label
    result["similar_documents"] = [
        f"{df.iloc[i]['doc_id']} ({df.iloc[i]['doc_type']}): {df.iloc[i]['text'][:90]}..."
        for i, _ in search_hybrid(text, n_similar)]
    return result

def ask(question):
    r = rag_answer(question)
    sources = "\n".join(f"[{i+1}] {d['doc_id']} ({d['doc_type']}): {d['text'][:110]}..."
                        for i, d in enumerate(r["docs"]))
    return r["answer"], sources

# ---------- email handling ----------
def email_to_text(subject, body):
    """Join Subject and Body the same way the corpus writes supplier emails ('Subject: ... . body')."""
    subject, body = (subject or "").strip().rstrip("."), (body or "").strip()
    return f"Subject: {subject}. {body}" if subject else body

def read_eml(raw_bytes):
    """Parse a .eml file into (subject, sender, date, plain-text body)."""
    msg = email.message_from_bytes(raw_bytes, policy=policy.default)
    part = msg.get_body(preferencelist=("plain", "html"))
    body = part.get_content() if part is not None else ""
    return str(msg["subject"] or ""), str(msg["from"] or ""), str(msg["date"] or ""), body

def load_uploaded(path):
    """Read one uploaded file and return a list of (name, text) documents."""
    path = getattr(path, "name", path)                           # works for str paths and file objects
    name = os.path.basename(path)
    ext = os.path.splitext(name)[1].lower()
    if ext == ".eml":
        subj, sender, date, body = read_eml(open(path, "rb").read())
        return [(name, email_to_text(subj, body))]
    if ext == ".csv":                                            # one document per row
        table = pd.read_csv(path)
        wanted = [c for c in table.columns if c.lower() in ("text", "body", "email", "message", "content")]
        col = wanted[0] if wanted else next((c for c in table.columns if table[c].dtype == object), table.columns[0])
        sub = [c for c in table.columns if c.lower() == "subject"]
        rows = []
        for i, r in table.head(200).iterrows():
            body = str(r[col])
            rows.append((f"{name} row {i + 1}", email_to_text(r[sub[0]] if sub else "", body)))
        return rows
    return [(name, open(path, encoding="utf-8", errors="ignore").read())]   # .txt and anything else

# ---------- 2. quick check without the UI ----------
sample = ("Purchase Order 5521: Request for 120 units of hydraulic pump HP 200. Supplier: TechSupply Global. "
          "Delivery required by 2025-03-14. URGENT: Equipment downtime impacting operations. Expedite delivery.")
print(json.dumps(process_document(sample), indent=2, default=str))

# ---------- 3. web demo (Gradio) ----------
import gradio as gr

def format_result(r):
    lines = [f"**Document type:** {r['document_type']} (margin {r['classifier_margin']})",
             f"**Urgency:** {r['urgency']}"]
    if "sentiment" in r:
        lines.append(f"**Sentiment:** {r['sentiment']}")
    ents = "\n".join(f"- {k}: {', '.join(map(str, v))}" for k, v in r["entities"].items()) or "- none found"
    lines.append("**Entities:**\n" + ents)
    lines.append("**Similar documents:**\n" + "\n".join(f"- {s}" for s in r["similar_documents"]))
    return "\n\n".join(lines)

def analyse_email_ui(subject, sender, body):
    text = email_to_text(subject, body)
    if not text.strip():
        return "Please enter an email body or document text."
    head = f"**From:** {sender}\n\n" if sender else ""
    return head + format_result(process_document(text))

def batch_ui(files):
    if not files:
        return pd.DataFrame({"info": ["Upload one or more .txt, .eml or .csv files."]})
    rows = []
    for f in (files if isinstance(files, list) else [files]):
        for name, text in load_uploaded(f):
            if not text.strip():
                continue
            r = process_document(text, n_similar=1)
            e = r["entities"]
            rows.append({"document": name, "type": r["document_type"], "urgency": r["urgency"],
                         "sentiment": r.get("sentiment", ""),
                         "parts": ", ".join(e.get("PART", [])), "suppliers": ", ".join(e.get("SUPPLIER", [])),
                         "quantity": ", ".join(map(str, e.get("QUANTITY", []))), "dates": ", ".join(e.get("DATE", [])),
                         "sites": ", ".join(e.get("SITE", []))})
    return pd.DataFrame(rows)

examples = [
    ["Order Update", "sales@partspro.example",
     "The delivery of bearing assembly 6205 2RS from PartsPro International has been delayed by 2 weeks. "
     "Raw material shortage has affected production."],
    ["Urgent: PO 7001", "buyer@company.example",
     "Purchase Order 7001: Request for 80 units of gearbox GB 300. Supplier: SpareMax Solutions. "
     "Delivery required by 2025-06-10. CRITICAL: Production line halted. Immediate shipment required."],
    ["Breakdown at Abu Dhabi", "maintenance@company.example",
     "Packaging Line PL 5 stopped at Abu Dhabi Warehouse. Cause: bearing seizure. "
     "Replacement of compressor unit CU 450 is required."],
]
questions = ["Are there any delays from suppliers affecting hydraulic pumps?",
             "What is the status of bearing assembly orders?",
             "Who won the football world cup?"]                  # off-topic: should be refused

with gr.Blocks(title="Procurement Document Intelligence") as demo:
    gr.Markdown("# Procurement Document Intelligence\n"
                "Analyse emails and documents, process a whole batch, or ask questions about the document repository.")
    with gr.Tab("Analyse an email / document"):
        with gr.Row():
            subj = gr.Textbox(label="Subject (optional)", value=examples[0][0])
            frm = gr.Textbox(label="From (optional)", value=examples[0][1])
        bod = gr.Textbox(lines=5, label="Email body or document text", value=examples[0][2])
        out = gr.Markdown()
        gr.Button("Analyse", variant="primary").click(analyse_email_ui, [subj, frm, bod], out)
        gr.Examples(examples=examples, inputs=[subj, frm, bod], label="Try an example")
    with gr.Tab("Batch upload"):
        gr.Markdown("Upload several **.txt**, **.eml** or **.csv** files (a CSV is read one document per row, "
                    "using a column named text, body, email or message). The table shows the result for each document.")
        files = gr.File(label="Documents", file_count="multiple", file_types=[".txt", ".eml", ".csv"])
        table = gr.Dataframe(label="Results")
        gr.Button("Process all", variant="primary").click(batch_ui, files, table)
    with gr.Tab("Ask the documents"):
        q = gr.Textbox(label="Question", value=questions[0])
        ans = gr.Textbox(label="Answer", lines=4)
        src = gr.Textbox(label="Sources", lines=6)
        gr.Button("Ask", variant="primary").click(ask, q, [ans, src])
        gr.Examples(examples=[[x] for x in questions], inputs=[q], label="Try a question")

demo.launch(share=True)

{
  "document_type": "Purchase Order",
  "classifier_margin": 1.97,
  "entities": {
    "PART": [
      "hydraulic pump hp 200"
    ],
    "SUPPLIER": [
      "Techsupply Global"
    ],
    "QUANTITY": [
      "120"
    ],
    "DATE": [
      "2025-03-14"
    ]
  },
  "urgency": "HIGH",
  "similar_documents": [
    "PO-0052 (Purchase Order): PO 5976: Request for 492 units of hydraulic pump HP 200. Supplier: QualityParts Asia. Deli...",
    "PO-0054 (Purchase Order): Purchase Order 7949: Procurement of 310x hydraulic pump HP 200 from TechSupply Global. Exp...",
    "PO-0037 (Purchase Order): PO 2122: Request for 40 units of hydraulic pump HP 200. Supplier: PartsPro International. ..."
  ]
}
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a275d8dee29bbc0045.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the t

## 8. Performance Summary Dashboard

This section presents a consolidated view of the system's performance
across all four modules.

In [13]:
# ============================================================
# SECTION 8: FINAL PERFORMANCE SUMMARY
# Every number is read from the variables of the cells above, so it always matches the notebook.
# ============================================================
import torch

print("=" * 66)
print("  SYSTEM PERFORMANCE SUMMARY")
print("=" * 66)

# ---- environment ----
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
print(f"\n  Hardware:        {gpu}")
print(f"  Embedding model: {EMB_MODEL}")
print(f"  Language model:  {MODEL_NAME}")
print(f"  Dataset:         {len(df)} synthetic documents, 4 categories, "
      f"{int(df['is_noisy'].sum())} with OCR noise")

# ---- Module 1 ----
print("\n  MODULE 1 - Document classification (accuracy)")
print(f"  {'Model':<22}{'5-fold CV':>12}{'Unseen templates':>20}{'Noisy docs':>13}")
for n, r in results.items():
    mark = "  <- best" if n == best_name else ""
    print(f"  {n:<22}{r['cv_mean']*100:>11.1f}%{r['unseen']*100:>19.1f}%{r['noisy']*100:>12.1f}%{mark}")

# ---- Module 2 ----
tc = prf(*tot_c); tn = prf(*tot_n)
print("\n  MODULE 2 - Named entity recognition (micro precision / recall / F1)")
print(f"  Clean text : P {tc[0]:.3f}  R {tc[1]:.3f}  F1 {tc[2]:.3f}")
print(f"  Noisy text : P {tn[0]:.3f}  R {tn[1]:.3f}  F1 {tn[2]:.3f}   (n={len(noisy_docs)} documents)")
print(f"  Urgency accuracy on purchase orders: clean {u_clean*100:.1f}% | noisy {u_noisy*100:.1f}%")

# ---- Module 3 ----
print("\n  MODULE 3 - Sentiment (keyword rules, supplier communications)")
print(f"  Accuracy: {accuracy:.1f}% over {len(supplier_docs)} emails")

# ---- Module 4 ----
print("\n  MODULE 4 - Retrieval (30 queries)")
print(f"  {'Method':<22}{'Hit@1':>8}{'Hit@3':>8}{'MRR':>8}{'P@5':>8}")
for name, s in all_scores.items():
    print(f"  {name:<22}{s.hit1.mean():>8.3f}{s.hit3.mean():>8.3f}{s.mrr.mean():>8.3f}{s.p5.mean():>8.3f}")

print("\n  MODULE 4 - Generated answers (LLM, retrieval-grounded)")
print(f"  Refusal gate (min dense similarity): {MIN_DENSE_SIM}")
print(f"  In-scope queries ({len(res)}): wrongly refused {res.refused.mean()*100:.1f}% | "
      f"entity grounding {np.nanmean(res.grounded)*100:.1f}% | "
      f"citation precision {np.nanmean(res.cite_precision)*100:.1f}%")
print(f"  Off-topic refused: {np.mean(off_refused)*100:.1f}% (n={len(off_topic)}) | "
      f"in-domain unanswerable refused: {np.mean(un_refused)*100:.1f}% (n={len(unanswerable)})")

print("\n  Note: data is synthetic and template-based; scores show feasibility, not real-world accuracy.")

  SYSTEM PERFORMANCE SUMMARY

  Hardware:        Tesla T4
  Embedding model: sentence-transformers/all-MiniLM-L6-v2
  Language model:  Qwen/Qwen2.5-3B-Instruct
  Dataset:         550 synthetic documents, 4 categories, 137 with OCR noise

  MODULE 1 - Document classification (accuracy)
  Model                    5-fold CV    Unseen templates   Noisy docs
  Logistic Regression         100.0%               86.2%       100.0%
  Linear SVM                  100.0%               91.7%       100.0%  <- best
  Naive Bayes                  99.6%               86.2%        99.3%
  Random Forest               100.0%               82.6%       100.0%
  Gradient Boosting            99.3%               56.0%        98.5%

  MODULE 2 - Named entity recognition (micro precision / recall / F1)
  Clean text : P 1.000  R 1.000  F1 1.000
  Noisy text : P 0.989  R 0.853  F1 0.916   (n=137 documents)
  Urgency accuracy on purchase orders: clean 100.0% | noisy 100.0%

  MODULE 3 - Sentiment (keyword rules, sup

## 9. Conclusion, Limitations and Production Roadmap

### What was achieved
A complete, GPU-accelerated NLP pipeline for procurement documents: classification, entity extraction, sentiment and urgency, hybrid retrieval, grounded LLM answers with refusal, and an interactive Gradio demo. Every module is scored against gold labels, and the results are reported honestly, including where they are weaker (unseen-template classification, noisy-text entity recall, sentiment ties).

### Ethical considerations addressed in the prototype
- **Hallucination control:** refusal gate, grounding check and mandatory citations.
- **Privacy:** synthetic data only; no real supplier or personal data.
- **Transparency:** every answer lists its source documents so a human can verify it.
- **Human-in-the-loop:** outputs support procurement staff and do not replace approval decisions.

### Limitations
- Data is synthetic and template-based, so real-world accuracy will be lower.
- Test sets are small (30 retrieval queries, 150 sentiment emails), so results carry uncertainty.
- Rule-based NER and keyword sentiment cannot handle unseen phrasing or sarcasm.
- English only; no scanned-image OCR (noise is simulated).
- The refusal threshold was calibrated on the same queries it was evaluated on.
- Entity grounding does not catch wrong relationships between correct entities.

### Production roadmap
1. Replace rule-based NER with a fine-tuned transformer NER model.
2. Replace keyword sentiment with a transformer classifier on GPU.
3. Add real OCR for scanned documents.
4. Move FAISS to a managed vector database.
5. Tune hybrid retrieval weights per query type on a separate validation set.
6. Fine-tune or upgrade the LLM, and validate on real, anonymised documents.
7. Add access control and audit logging.